In [38]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\toolestown"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\toolestown\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\toolestown\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Toolestown - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Toolestown - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Toolestown - Eletromecânico.dwg")

conn = load_to_duckdb(dxf_file, "PV Toolestown")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

1
Project PV Toolestown already exists
Removed existing data for project: PV Toolestown
Successfully loaded project
Loaded 4643 entities for project: PV Toolestown
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [39]:
data_ELM = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP, STR, rotation FROM cad_entities WHERE project_name = 'PV Toolestown'")

print(data_ELM)

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count
""")

┌───────────────┬───────────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬──────────┬─────────┬───────────┐
│     layer     │         name          │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP    │   STR   │ rotation  │
│    varchar    │        varchar        │   float   │   float   │ varchar │ varchar │ varchar │ varchar  │ varchar │   float   │
├───────────────┼───────────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼──────────┼─────────┼───────────┤
│ EMF_em_PCS_04 │ EMF422-2_eletrocentro │  692392.5 │  735909.3 │ 0       │ 04      │ NULL    │ NULL     │ NULL    │     180.0 │
│ EMF_em_PCS_03 │ EMF422-2_eletrocentro │ 692593.25 │ 735954.94 │ 0       │ 03      │ NULL    │ NULL     │ NULL    │     180.0 │
│ EMF_em_PCS_02 │ EMF422-2_eletrocentro │  692794.5 │  736116.5 │ 0       │ 02      │ NULL    │ NULL     │ NULL    │       0.0 │
│ EMF_em_PCS_01 │ EMF422-2_eletrocentro │ 692938.75 │  736170.9 │ 0       │ 01      │ NULL    │ N

In [40]:
strings = conn.sql("SELECT * FROM data_ELM WHERE name LIKE 'EMF422-1_string_%'")

strings_df = strings.fetchdf()
print(strings)
print(strings_df.describe())
print(strings_df.info())

┌───────────────┬───────────────────┬──────────┬───────────┬─────────┬─────────┬─────────┬──────────┬─────────┬──────────┐
│     layer     │       name        │    X     │     Y     │   UFV   │   PST   │   INV   │   GRP    │   STR   │ rotation │
│    varchar    │      varchar      │  float   │   float   │ varchar │ varchar │ varchar │ varchar  │ varchar │  float   │
├───────────────┼───────────────────┼──────────┼───────────┼─────────┼─────────┼─────────┼──────────┼─────────┼──────────┤
│ EMF_em_PCS_01 │ EMF422-1_string_2 │ 692592.9 │  736341.9 │ 00      │ 01      │ 01      │ 00.01.01 │ 01      │      0.0 │
│ EMF_em_PCS_01 │ EMF422-1_string_2 │ 692592.9 │  736341.9 │ 00      │ 01      │ 01      │ 00.01.01 │ 02      │    180.0 │
│ EMF_em_PCS_01 │ EMF422-1_string_3 │ 692598.7 │  736334.5 │ 00      │ 01      │ 01      │ 00.01.01 │ 03      │      0.0 │
│ EMF_em_PCS_01 │ EMF422-1_string_3 │ 692598.7 │  736332.1 │ 00      │ 01      │ 01      │ 00.01.01 │ 04      │      0.0 │
│ EMF_em_PCS_01 

In [41]:
inverter = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP FROM data_ELM WHERE name = 'EMF422-2_inversor'")
print(inverter)

inverter_df = inverter.fetchdf()
inverter_df["UFV"] = inverter_df["UFV"].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + "." + inverter_df["INV"].astype(str)
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)

print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

┌───────────────┬───────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┐
│     layer     │       name        │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │
│    varchar    │      varchar      │   float   │   float   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼───────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┤
│ EMF_em_PCS_04 │ EMF422-2_inversor │  692321.3 │  735905.9 │ 0       │ 04      │ 01      │         │
│ EMF_em_PCS_04 │ EMF422-2_inversor │  692339.6 │  735857.1 │ 0       │ 04      │ 05      │         │
│ EMF_em_PCS_04 │ EMF422-2_inversor │ 692349.25 │  735827.9 │ 0       │ 04      │ 08      │         │
│ EMF_em_PCS_04 │ EMF422-2_inversor │  692358.8 │  735798.6 │ 0       │ 04      │ 10      │         │
│ EMF_em_PCS_04 │ EMF422-2_inversor │ 692373.25 │  735779.1 │ 0       │ 04      │ 12      │         │
│ EMF_em_PCS_04 │ EMF422-2_inversor │ 692376.44 │  735759.6 │ 0       │ 04      │ 

In [42]:
for idx, row in strings_df.iterrows():
    x = row['X']
    y = row['Y']


    mask_TMB = (
        (strings_df['Y'] >= y - 5) &
        (strings_df['Y'] <= y + 5) &
        (strings_df['X'] >= x - 0.5) &
        (strings_df['X'] <= x + 0.5) &
        (strings_df['Y'] != y) &
        (strings_df.index != idx)
    )

    if row['name'] == 'EMF422-1_string_2':
        if row['rotation'] == 0:
            strings_df.loc[idx, 'Type'] = "U_R"
        else:
            strings_df.loc[idx, 'Type'] = "U_L"

    elif mask_TMB.any():
        y_neighbors = strings_df.loc[mask_TMB, 'Y']

        if (y > y_neighbors).all():
            strings_df.loc[idx, 'Type'] = "T"
        elif (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "M"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "C"


strings_df.loc[:,'Line'] = np.where(
    (strings_df["Type"] == "C") | (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L") | (strings_df["Type"] == "M"),
    round(strings_df["Y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["Y"] - 2.3204),
        round(strings_df["Y"] + 2.3204)
    )
)


strings_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)
print(strings_df)
print(strings_df['Type'].value_counts())

              layer               name            X            Y UFV PST INV  \
0     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
1     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
2     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
3     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
4     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736334.6250  00  01  03   
...             ...                ...          ...          ...  ..  ..  ..   
4451  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734020.4375  00  11  18   
4452  EMF_em_PCS_11  EMF422-1_string_1  695373.9375  734013.0000  00  11  18   
4453  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734015.3125  00  11  18   
4454  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734013.0000  00  11  18   
4455  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734010.6875  00  11  18   

           GRP STR  rotation Type      

In [43]:
#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure
condition2 = abs(strings_df["X"] - strings_df["X"].shift()) > 40

#Different PSTs
condition3 = strings_df["GRP"] != strings_df["GRP"].shift()



break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition2

strings_df['SubGroup'] = break_condition_1.cumsum()
strings_df['Group'] = break_condition_2.cumsum()
print(strings_df)

              layer               name            X            Y UFV PST INV  \
0     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
1     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
2     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
3     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
4     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736334.6250  00  01  03   
...             ...                ...          ...          ...  ..  ..  ..   
4451  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734020.4375  00  11  18   
4452  EMF_em_PCS_11  EMF422-1_string_1  695373.9375  734013.0000  00  11  18   
4453  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734015.3125  00  11  18   
4454  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734013.0000  00  11  18   
4455  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734010.6875  00  11  18   

           GRP STR  rotation Type      

In [44]:
strings_df['W exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['X'] - 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] - 11.053,
        strings_df['X'] - 16.913
    )
)

strings_df['E exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['X'] + 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] + 11.053,
        strings_df['X'] + 16.913
    )
)

strings_df['SubGroup - Position X - W'] = strings_df.groupby("SubGroup")["W exit"].transform('min')
strings_df['SubGroup - Position X - E'] = strings_df.groupby("SubGroup")["E exit"].transform('max')

strings_df['Group - Position X - W'] = strings_df.groupby("Group")["W exit"].transform('min')
strings_df['Group - Position X - E'] = strings_df.groupby("Group")["E exit"].transform('max')

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['name'].transform('count')

print(strings_df)
print(strings_df['No. of Strings'].value_counts())
print(strings_df['No. of Strings'].value_counts().sum())

              layer               name            X            Y UFV PST INV  \
0     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
1     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736344.3750  00  01  03   
2     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
3     EMF_em_PCS_01  EMF422-1_string_2  692592.8750  736341.8750  00  01  01   
4     EMF_em_PCS_01  EMF422-1_string_2  692931.6875  736334.6250  00  01  03   
...             ...                ...          ...          ...  ..  ..  ..   
4451  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734020.4375  00  11  18   
4452  EMF_em_PCS_11  EMF422-1_string_1  695373.9375  734013.0000  00  11  18   
4453  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734015.3125  00  11  18   
4454  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734013.0000  00  11  18   
4455  EMF_em_PCS_11  EMF422-1_string_3  695397.3750  734010.6875  00  11  18   

           GRP STR  rotation  ...      

In [45]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Line', 'SubGroup - Position X - W', 'SubGroup - Position X - E', 'Group - Position X - W', 'Group - Position X - E', 'No. of Strings']].first()
groups_df.rename(columns={'Line': 'Y', 'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 22

               GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT  \
SubGroup                                                              
1         00.01.03  736344.0  692920.6250  692942.7500  692920.6250   
2         00.01.01  736342.0  692581.8125  692603.9375  692581.8125   
3         00.01.03  736335.0  692920.6250  692942.7500  692920.6250   
4         00.01.01  736332.0  692581.7500  692627.3750  692581.7500   
5         00.01.03  736325.0  692914.8125  692936.9375  692914.8125   
...            ...       ...          ...          ...          ...   
643       00.11.17  734042.0  695265.8125  695416.8750  695265.8125   
644       00.11.17  734032.0  695277.5000  695311.3750  695277.5000   
645       00.11.18  734032.0  695277.5000  695416.8750  695277.5000   
646       00.11.18  734023.0  695321.8750  695414.3125  695321.8750   
647       00.11.18  734013.0  695368.7500  695414.3125  695368.7500   

           X-E-STRUCT  No. of Stri

Duct_map is needed to represent how many ducts are necessary and how many strings are there in each duct. Given that the maximum of strings in a duct is 3, the duct_map is as follows:
=

In [46]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
    21: '-3-3-3-3-3-3-3-',
    22: '-3-3-3-3-3-3-3-1-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['W or E'] = np.where(
    abs(groups_df['X-W-STRUCT'] - groups_df['inverter_x']) > abs(groups_df['X-E-STRUCT'] - groups_df['inverter_x']),
    "E",
    "W"
)

groups_df['closest_x'] = np.where(
    groups_df['W or E'] == "W",
    groups_df['X-W-STRUCT'],
    groups_df['X-E-STRUCT']
)
print(groups_df)

          GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT   X-E-STRUCT  \
0    00.01.03  736344.0  692920.6250  692942.7500  692920.6250  692942.7500   
1    00.01.01  736342.0  692581.8125  692603.9375  692581.8125  692603.9375   
2    00.01.03  736335.0  692920.6250  692942.7500  692920.6250  692942.7500   
3    00.01.01  736332.0  692581.7500  692627.3750  692581.7500  692627.3750   
4    00.01.03  736325.0  692914.8125  692936.9375  692914.8125  692936.9375   
..        ...       ...          ...          ...          ...          ...   
642  00.11.17  734042.0  695265.8125  695416.8750  695265.8125  695416.8750   
643  00.11.17  734032.0  695277.5000  695311.3750  695277.5000  695416.8750   
644  00.11.18  734032.0  695277.5000  695416.8750  695277.5000  695416.8750   
645  00.11.18  734023.0  695321.8750  695414.3125  695321.8750  695414.3125   
646  00.11.18  734013.0  695368.7500  695414.3125  695368.7500  695414.3125   

     No. of Strings Duct Definition  Number of Duct

Delta X Delta Y on the structure coordinates and the inverter and creating the final Dataframe for the Duct Script
=

In [51]:
script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_x', 'Y', 'inverter_x', 'inverter_y']].copy()
script_df.rename(columns={'closest_x': 'X'}, inplace=True)

script_df['intermediary_y_1'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 1.7,
    script_df['inverter_y'] + 1.7
)

script_df['intermediary_y_2'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 1.5,
    script_df['inverter_y'] + 1.5
)

script_df['intermediary_y_3'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 1.3,
    script_df['inverter_y'] + 1.3
)

script_df['inverter_y'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 0.1,
    script_df['inverter_y'] + 0.1
)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y_1']} {row['inverter_x']},{row['intermediary_y_2']} {row['inverter_x']},{row['intermediary_y_3']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas - Toolestown.scr", 'w') as f:
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y_1']} {row['inverter_x']},{row['intermediary_y_2']} {row['inverter_x']},{row['intermediary_y_3']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y_1']} {row['inverter_x']},{row['intermediary_y_2']} {row['inverter_x']},{row['intermediary_y_3']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: (f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y_1']} {row['inverter_x']},{row['intermediary_y_2']} {row['inverter_x']},{row['intermediary_y_3']} {row['inverter_x']},{row['inverter_y']} " + "\n") * str(row['Duct Definition']).count('3'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Dutos - Toolestown.scr", 'w') as f:
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write(''.join(layer3_commands) + '\n')

0      _.pline 692920.625,736344.0 692893.25,736344.0...
1      _.pline 692603.9375,736342.0 692661.375,736342...
2      _.pline 692920.625,736335.0 692893.25,736335.0...
3      _.pline 692627.375,736332.0 692661.375,736332....
4      _.pline 692914.8125,736325.0 692893.25,736325....
                             ...                        
642    _.pline 695265.8125,734042.0 695267.0,734042.0...
643    _.pline 695277.5,734032.0 695267.0,734032.0 69...
644    _.pline 695277.5,734032.0 695323.0,734032.0 69...
645    _.pline 695321.875,734023.0 695323.0,734023.0 ...
646    _.pline 695368.75,734013.0 695323.0,734013.0 6...
Name: Script Valas, Length: 647, dtype: str
